# Cell2Sentence (C2S) — Google Colab Benchmarking Notebook

This notebook runs the **Cell2Sentence** cell-type annotation tool inside the
full cross-validation benchmarking framework used in the thesis.

## Quick-start
1. **Runtime ▸ Change runtime type ▸ GPU** (T4 or A100 recommended)
2. **Run all cells top-to-bottom** (`Runtime ▸ Run all`)
3. Set `DATA_PATH` and `RESULTS_DIR` in the *Configuration* cell below

## Execution modes (select in the *Run benchmarking* cell)
| Mode | Function | Use case |
|------|----------|----------|
| K-fold CV (default) | `run_all_tools_cv()` | Standard benchmarking |
| Single 80/20 split | `run_all_tools_single_split()` | Simulated / replicate datasets |
| Separate datasets | `run_separate_datasets()` | Cross-dataset generalisation |


In [ ]:
# Install dependencies
# Run once per Colab session; restart runtime if prompted to do so.
!pip install -q cell2sentence==1.1.0 memory_profiler scanpy anndata
print("Installation complete.")


In [ ]:
# ── Patch cell2sentence ──────────────────────────────────────────────────
# Must run BEFORE any `import cell2sentence`. Two patches, both idempotent:
#
#   (1) transformers 5.x rename:
#       Trainer(tokenizer=...) was deprecated and removed in transformers 5.x
#       in favor of Trainer(processing_class=...). cell2sentence 1.1.0 still
#       uses the old keyword.
#
#   (2) Persist fine-tuned weights:
#       cell2sentence.csmodel.CSModel.fine_tune() trains the model in memory
#       but NEVER saves it back to self.save_path. Then
#       predict_cell_types_of_data() reloads from self.save_path — i.e., it
#       reloads the ORIGINAL pretrained weights and silently discards all
#       fine-tuning. We inject trainer.save_model() + tokenizer.save_pretrained()
#       immediately after trainer.train() to fix this.
#
# Re-running this cell is safe (idempotent).
import site, os

_PERSIST_MARKER = "# c2s_ft: persist fine-tuned weights"

_patched_any = False
for _sp in site.getsitepackages():
    _csmodel_path = os.path.join(_sp, 'cell2sentence', 'csmodel.py')
    if not os.path.exists(_csmodel_path):
        continue

    with open(_csmodel_path, 'r') as _f:
        _content = _f.read()
    _orig = _content

    # ── Patch 1/2: transformers 5.x compatibility ───────────────────────
    if 'tokenizer=self.tokenizer' in _content:
        _content = _content.replace(
            'tokenizer=self.tokenizer',
            'processing_class=self.tokenizer'
        )
        print("  [patch 1/2] transformers 5.x: tokenizer -> processing_class  ✓")
    else:
        print("  [patch 1/2] already applied (or upstream changed)")

    # ── Patch 2/2: persist fine-tuned weights ───────────────────────────
    if _PERSIST_MARKER not in _content:
        _needle = (
            "        trainer.train()\n"
            "        print(f\"Finetuning completed."
        )
        _replacement = (
            "        trainer.train()\n"
            f"        {_PERSIST_MARKER}\n"
            "        trainer.save_model(self.save_path)\n"
            "        self.tokenizer.save_pretrained(self.save_path)\n"
            "        print(f\"Finetuning completed."
        )
        if _needle in _content:
            _content = _content.replace(_needle, _replacement)
            print("  [patch 2/2] persist fine-tuned weights: save_model added  ✓")
        else:
            print("  [patch 2/2] WARNING: anchor not found in csmodel.py")
            print("              fine-tuning will NOT persist. Inspect the file manually:")
            print(f"              {_csmodel_path}")
    else:
        print("  [patch 2/2] already applied")

    if _content != _orig:
        with open(_csmodel_path, 'w') as _f:
            _f.write(_content)
        print(f"  Patches written to {_csmodel_path}")
    else:
        print("  No changes needed.")

    _patched_any = True
    break

if not _patched_any:
    print("WARNING: csmodel.py not found — cell2sentence may not be installed yet.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── USER CONFIGURATION ──────────────────────────────────────────────────────
DRIVE_ROOT      = "/content/drive/MyDrive/Thesis FM Computing"
DATA_PATH        = f"{DRIVE_ROOT}/h5ad_real/Marques-Brain-2016_for_use_subset.h5ad"
RESULTS_DIR     = f"{DRIVE_ROOT}/c2s_results"

# Cross-validation settings (used by run_all_tools_cv)
CV_CONFIG = {
    'k': 3,
    'group_by': None,                    # set to a donor column for grouped CV
    'stratify_by': 'Ground_Truth_Celltype',
    'seed': 123,
}
# ────────────────────────────────────────────────────────────────────────────
import os
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Results will be saved to: {RESULTS_DIR}")


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import anndata as ad
import torch
import tempfile
import time
import warnings
import pickle
import os
import scipy.sparse as sp
from datetime import datetime
from typing import Dict, List, Tuple, Any, Optional, Callable
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import accuracy_score, confusion_matrix
from memory_profiler import memory_usage

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}, "
          f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM")


In [ ]:
# ── Benchmarking helper functions ──────────────────────────────────────────
# Ported verbatim from benchmarking_helpers.ipynb

fold_cache: Dict = {}   # global cache — clear between datasets


def calculate_metrics(predicted: List[str], true_labels: List[str]) -> Dict[str, Any]:
    unassigned_rate = np.sum(np.array(predicted) == 'Unknown') / len(predicted)
    predicted = np.array(predicted)
    true_labels = np.array(true_labels)
    overall_accuracy = accuracy_score(true_labels, predicted)
    cell_types = np.unique(np.concatenate([predicted, true_labels]))
    cell_types = cell_types[~pd.isnull(cell_types)]
    cell_types = cell_types[cell_types != 'Unknown']
    precision_scores = np.zeros(len(cell_types))
    recall_scores    = np.zeros(len(cell_types))
    f1_scores        = np.zeros(len(cell_types))
    for i, ct in enumerate(cell_types):
        TP = np.sum((predicted == ct) & (true_labels == ct))
        FP = np.sum((predicted == ct) & (true_labels != ct))
        FN = np.sum((predicted != ct) & (true_labels == ct))
        precision_scores[i] = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall_scores[i]    = TP / (TP + FN) if (TP + FN) > 0 else 0
        if precision_scores[i] + recall_scores[i] > 0:
            f1_scores[i] = 2 * (precision_scores[i] * recall_scores[i]) / (precision_scores[i] + recall_scores[i])
    macro_f1 = np.mean(f1_scores)
    n   = len(predicted)
    p_o = overall_accuracy
    p_e = sum((np.sum(predicted == ct) / n) * (np.sum(true_labels == ct) / n) for ct in cell_types)
    cohens_kappa = (p_o - p_e) / (1 - p_e) if (1 - p_e) > 0 else 1.0
    ct_to_idx = {ct: i for i, ct in enumerate(cell_types)}
    n_ct = len(cell_types)
    C = np.zeros((n_ct, n_ct))
    for tc, pc in zip(true_labels, predicted):
        if tc in ct_to_idx and pc in ct_to_idx:
            C[ct_to_idx[tc], ct_to_idx[pc]] += 1
    c = np.trace(C); s = C.sum(); t_k = C.sum(axis=1); p_k = C.sum(axis=0)
    num = c * s - np.dot(t_k, p_k)
    den = np.sqrt((s**2 - np.dot(p_k, p_k)) * (s**2 - np.dot(t_k, t_k)))
    mcc = num / den if den > 0 else 0.0
    rare_idx = np.argmin([np.sum(true_labels == ct) for ct in cell_types])
    return {
        'overall_accuracy': overall_accuracy,
        'macro_f1': macro_f1,
        'cohens_kappa': cohens_kappa,
        'mcc': mcc,
        'unassigned_rate': unassigned_rate,
        'rare_type_f1': f1_scores[rare_idx],
        'rare_type_name': cell_types[rare_idx],
        'per_class_metrics': pd.DataFrame({
            'cell_type': cell_types,
            'precision': precision_scores,
            'recall': recall_scores,
            'f1': f1_scores
        })
    }


def create_cv_folds(adata: ad.AnnData, k: int = 5, group_by: Optional[str] = None,
                    stratify_by: str = 'Ground_Truth_Celltype', seed: int = 123) -> List[List[str]]:
    np.random.seed(seed)
    fold_cell_lists = []
    if group_by is not None and group_by in adata.obs.columns:
        print(f"Mode: Grouped K-Fold CV by '{group_by}'.")
        all_groups = adata.obs[group_by].unique()
        if len(all_groups) < k:
            raise ValueError('Number of groups is less than k.')
        shuffled_groups = np.random.permutation(all_groups)
        group_folds = np.array_split(shuffled_groups, k)
        for i in range(k):
            test_mask = adata.obs[group_by].isin(group_folds[i])
            fold_cell_lists.append(adata.obs.index[test_mask].tolist())
    else:
        print(f"Mode: Stratified K-Fold CV by '{stratify_by}'.")
        if stratify_by not in adata.obs.columns:
            raise ValueError('Stratification variable not found in metadata.')
        skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
        stratify_labels = adata.obs[stratify_by]
        valid_mask = stratify_labels.notna()
        n_dropped = int((~valid_mask).sum())
        if n_dropped > 0:
            print(f"Warning: Dropping {n_dropped} cells with NaN '{stratify_by}'")
        valid_indices = np.where(valid_mask)[0]
        valid_labels  = stratify_labels.iloc[valid_indices]
        for _, test_idx in skf.split(valid_indices, valid_labels):
            fold_cell_lists.append(adata.obs.index[valid_indices[test_idx]].tolist())
    print('Fold creation complete.')
    for i, fc in enumerate(fold_cell_lists):
        print(f'  Fold {i+1}: {len(fc)} cells')
    return fold_cell_lists


def _is_log_normalized(adata_obj):
    if 'log1p' in adata_obj.uns:
        return True, 'log1p flag found in .uns'
    max_val = adata_obj.X.max()
    if max_val < 20:
        return True, f'max value ({max_val:.2f}) suggests log-normalised data'
    if sp.issparse(adata_obj.X):
        sample = adata_obj.X.data[:min(10000, len(adata_obj.X.data))]
    else:
        sample = adata_obj.X.ravel()[:min(10000, adata_obj.X.size)]
    int_fraction = np.mean(sample == np.floor(sample))
    if int_fraction < 0.1:
        return True, f'only {int_fraction*100:.1f}% integers suggests log-normalised'
    if max_val > 100:
        return False, f'max value ({max_val:.2f}) suggests raw counts'
    return False, 'uncertain — defaulting to raw counts assumption'


def get_fold_data_cached(adata: ad.AnnData, folds: List[List[str]],
                         fold_index: int,
                         label_key: str = 'Ground_Truth_Celltype') -> Dict[str, Any]:
    global fold_cache
    cache_key = f'fold_{fold_index}'
    if cache_key not in fold_cache:
        print(f'    Computing fold {fold_index} data and markers...')
        test_cells  = folds[fold_index]
        train_cells = [c for c in adata.obs_names if c not in test_cells]
        adata_train = adata[train_cells].copy()
        adata_test  = adata[test_cells].copy()
        print(f'    Training: {adata_train.n_obs} cells, Testing: {adata_test.n_obs} cells')
        is_norm, reason = _is_log_normalized(adata_train)
        print(f'    Normalization check: {reason}')
        if not is_norm:
            print('    Log-normalizing...')
            sc.pp.normalize_total(adata_train, target_sum=1e4)
            sc.pp.log1p(adata_train)
            sc.pp.normalize_total(adata_test, target_sum=1e4)
            sc.pp.log1p(adata_test)
        sc.tl.rank_genes_groups(adata_train, groupby=label_key, method='t-test', pts=True)
        groups = adata_train.obs[label_key].unique()
        markers_list = []
        for group in groups:
            scores = adata_train.uns['rank_genes_groups']['scores'][group]
            pvals  = adata_train.uns['rank_genes_groups']['pvals'][group]
            logfc  = adata_train.uns['rank_genes_groups']['logfoldchanges'][group]
            genes  = adata_train.uns['rank_genes_groups']['names'][group]
            markers_list.append(pd.DataFrame({
                'gene': genes, 'cluster': group,
                'avg_logFC': logfc, 'p_val': pvals, 'score': scores
            }))
        markers = pd.concat(markers_list, ignore_index=True)
        print(f'    Found {markers.shape[0]} marker genes across {len(groups)} cell types')
        fold_cache[cache_key] = {
            'train': adata_train, 'test': adata_test, 'markers': markers,
            'fold_info': {
                'train_cells': adata_train.n_obs, 'test_cells': adata_test.n_obs,
                'train_cell_types': adata_train.obs[label_key].value_counts(),
                'test_cell_types':  adata_test.obs[label_key].value_counts()
            }
        }
    else:
        print(f'    Using cached fold {fold_index} data')
    return fold_cache[cache_key]


def aggregate_tool_results(all_predictions, all_true_labels, all_confidence_scores,
                           all_cell_ids, fold_metrics, fold_runtimes,
                           fold_peak_memories, fold_peak_vrams, tool_name):
    valid_fm  = [fm for fm in fold_metrics if fm is not None]
    valid_rt  = [rt for rt in fold_runtimes if not pd.isnull(rt)]
    valid_mem = [m  for m  in fold_peak_memories if m is not None and not pd.isnull(m)]
    pooled = calculate_metrics(all_predictions, all_true_labels)
    pooled['confusion_matrix'] = confusion_matrix(all_true_labels, all_predictions)
    if valid_fm:
        def _ci(vals):
            return (np.mean(vals) + np.array([-1.96, 1.96]) *
                    (np.std(vals, ddof=1) / np.sqrt(len(vals)))).tolist() if len(vals) > 1 else [np.mean(vals)]*2
        def _std(vals):
            return np.std(vals, ddof=1) if len(vals) > 1 else 0
        fa = [fm['overall_accuracy']  for fm in valid_fm]
        ff = [fm['macro_f1']          for fm in valid_fm]
        fk = [fm['cohens_kappa']      for fm in valid_fm]
        fm_ = [fm.get('mcc', np.nan)  for fm in valid_fm]
        fr = [fm.get('rare_type_f1', np.nan) for fm in valid_fm]
        fu = [fm.get('unassigned_rate', np.nan) for fm in valid_fm]
        fold_variation = {
            'accuracy_mean': np.mean(fa), 'accuracy_std': _std(fa),
            'accuracy_folds': fa, 'accuracy_ci': _ci(fa),
            'f1_mean': np.mean(ff), 'f1_std': _std(ff),
            'f1_folds': ff, 'f1_ci': _ci(ff),
            'kappa_mean': np.mean(fk), 'kappa_std': _std(fk),
            'kappa_folds': fk, 'kappa_ci': _ci(fk),
            'mcc_mean': np.nanmean(fm_), 'mcc_std': _std(fm_), 'mcc_folds': fm_,
            'rare_f1_mean': np.nanmean(fr), 'rare_f1_std': _std(fr), 'rare_f1_folds': fr,
            'unassigned_mean': np.nanmean(fu), 'unassigned_std': _std(fu), 'unassigned_folds': fu
        }
    else:
        fold_variation = {k: (np.nan if 'mean' in k or 'std' in k else [])
                          for k in ['accuracy_mean','accuracy_std','accuracy_folds','accuracy_ci',
                                    'f1_mean','f1_std','f1_folds','f1_ci',
                                    'kappa_mean','kappa_std','kappa_folds','kappa_ci',
                                    'mcc_mean','mcc_std','mcc_folds',
                                    'rare_f1_mean','rare_f1_std','rare_f1_folds',
                                    'unassigned_mean','unassigned_std','unassigned_folds']}
        fold_variation['accuracy_ci'] = [np.nan, np.nan]
        fold_variation['f1_ci']       = [np.nan, np.nan]
        fold_variation['kappa_ci']    = [np.nan, np.nan]
    valid_vrams = [v for v in fold_peak_vrams if v is not None and not pd.isnull(v) and v > 0]
    return {
        'tool_name': tool_name,
        'pooled_metrics': pooled,
        'fold_variation': fold_variation,
        'runtime_stats': {
            'mean_seconds': np.mean(valid_rt) if valid_rt else np.nan,
            'std_seconds':  np.std(valid_rt, ddof=1) if len(valid_rt) > 1 else 0,
            'total_seconds': np.sum(valid_rt) if valid_rt else np.nan,
            'fold_runtimes': valid_rt
        },
        'memory_stats': {
            'mean_mb': np.mean(valid_mem) if valid_mem else np.nan,
            'std_mb':  np.std(valid_mem, ddof=1) if len(valid_mem) > 1 else 0,
            'max_mb':  np.max(valid_mem) if valid_mem else np.nan,
            'min_mb':  np.min(valid_mem) if valid_mem else np.nan,
            'fold_memories': valid_mem
        },
        'vram_stats': {
            'mean_mb': np.mean(valid_vrams) if valid_vrams else np.nan,
            'std_mb':  np.std(valid_vrams, ddof=1) if len(valid_vrams) > 1 else 0,
            'max_mb':  np.max(valid_vrams) if valid_vrams else np.nan,
            'fold_vrams': valid_vrams
        },
        'detailed_results': {
            'predictions': all_predictions, 'true_labels': all_true_labels,
            'confidence_scores': all_confidence_scores, 'cell_ids': all_cell_ids
        },
        'num_successful_folds': len(valid_fm),
        'num_total_folds': len(fold_metrics)
    }


def run_single_tool_cv(adata: ad.AnnData, tool_function, folds: List[List[str]],
                       tool_name: str) -> Dict[str, Any]:
    print(f'\n=== Running {tool_name} ===')
    all_predictions, all_true_labels, all_confidence_scores, all_cell_ids = [], [], [], []
    fold_metrics, fold_runtimes, fold_peak_memories, fold_peak_vrams = [], [], [], []
    for i in range(len(folds)):
        print(f'  Fold {i+1}/{len(folds)}...')
        fold_data = get_fold_data_cached(adata, folds, i)
        try:
            fold_result  = tool_function(fold_data['train'], fold_data['test'], fold_data['markers'])
            runtime      = fold_result.get('method_walltime', np.nan)
            required     = ['predictions', 'true_labels', 'confidence_scores', 'cell_ids']
            if not all(k in fold_result for k in required):
                raise ValueError(f'Tool must return dict with: {required}')
            all_predictions.extend(fold_result['predictions'])
            all_true_labels.extend(fold_result['true_labels'])
            all_confidence_scores.extend(fold_result['confidence_scores'])
            all_cell_ids.extend(fold_result['cell_ids'])
            fm = calculate_metrics(fold_result['predictions'], fold_result['true_labels'])
            fold_metrics.append(fm)
            fold_runtimes.append(runtime)
            peak_mem  = fold_result.get('peak_memory_mb', None)
            peak_vram = fold_result.get('peak_vram_mb', None)
            fold_peak_memories.append(peak_mem)
            fold_peak_vrams.append(peak_vram)
            mem_str  = f', RAM: {peak_mem:.0f} MB'  if peak_mem  is not None else ''
            vram_str = f', VRAM: {peak_vram:.0f} MB' if peak_vram else ''
            print(f'    Done in {runtime:.2f}s, accuracy: {fm["overall_accuracy"]*100:.2f}%{mem_str}{vram_str}')
        except Exception as e:
            warnings.warn(f'{tool_name} failed on fold {i+1}: {e}')
            fold_metrics.append(None); fold_runtimes.append(np.nan)
            fold_peak_memories.append(None); fold_peak_vrams.append(None)
            print(f'    FAILED: {e}')
    return aggregate_tool_results(all_predictions, all_true_labels, all_confidence_scores,
                                  all_cell_ids, fold_metrics, fold_runtimes,
                                  fold_peak_memories, fold_peak_vrams, tool_name)


def aggregate_all_tools(all_tool_results: Dict[str, Dict]) -> Dict[str, Any]:
    rows = []
    for tool_name, result in all_tool_results.items():
        rows.append({
            'tool':                       tool_name,
            'pooled_accuracy':            result['pooled_metrics']['overall_accuracy'],
            'pooled_macro_f1':            result['pooled_metrics']['macro_f1'],
            'pooled_kappa':               result['pooled_metrics']['cohens_kappa'],
            'pooled_mcc':                 result['pooled_metrics']['mcc'],
            'pooled_rare_type_f1':        result['pooled_metrics']['rare_type_f1'],
            'pooled_unassigned_rate':     result['pooled_metrics']['unassigned_rate'],
            'accuracy_mean':              result['fold_variation']['accuracy_mean'],
            'accuracy_std':               result['fold_variation']['accuracy_std'],
            'accuracy_ci_lower':          result['fold_variation']['accuracy_ci'][0],
            'accuracy_ci_upper':          result['fold_variation']['accuracy_ci'][1],
            'f1_mean':                    result['fold_variation']['f1_mean'],
            'f1_std':                     result['fold_variation']['f1_std'],
            'f1_ci_lower':                result['fold_variation']['f1_ci'][0],
            'f1_ci_upper':                result['fold_variation']['f1_ci'][1],
            'kappa_mean':                 result['fold_variation']['kappa_mean'],
            'kappa_std':                  result['fold_variation']['kappa_std'],
            'kappa_ci_lower':             result['fold_variation']['kappa_ci'][0],
            'kappa_ci_upper':             result['fold_variation']['kappa_ci'][1],
            'mcc_mean':                   result['fold_variation']['mcc_mean'],
            'mcc_std':                    result['fold_variation']['mcc_std'],
            'rare_f1_mean':               result['fold_variation']['rare_f1_mean'],
            'rare_f1_std':                result['fold_variation']['rare_f1_std'],
            'unassigned_mean':            result['fold_variation']['unassigned_mean'],
            'unassigned_std':             result['fold_variation']['unassigned_std'],
            'runtime_mean':               result['runtime_stats']['mean_seconds'],
            'runtime_std':                result['runtime_stats']['std_seconds'],
            'peak_system_memory_mean_mb': result['memory_stats']['mean_mb'],
            'peak_system_memory_std_mb':  result['memory_stats']['std_mb'],
            'peak_system_memory_max_mb':  result['memory_stats']['max_mb'],
            'peak_GPU_memory_mean_mb':    result['vram_stats']['mean_mb'],
            'peak_GPU_memory_std_mb':     result['vram_stats']['std_mb'],
            'peak_GPU_memory_max_mb':     result['vram_stats']['max_mb'],
            'successful_folds':           result['num_successful_folds'],
        })
    df = pd.DataFrame(rows)
    if df.empty:
        warnings.warn('No tool results to aggregate.')
        return {'comparison_table': df, 'detailed_results': all_tool_results,
                'best_tool': None, 'summary_stats': {}}
    df = df.sort_values('pooled_kappa', ascending=False).reset_index(drop=True)
    return {
        'comparison_table': df,
        'detailed_results': all_tool_results,
        'best_tool': df.iloc[0]['tool'],
        'summary_stats': {
            'num_tools': len(df),
            'best_accuracy': df['pooled_accuracy'].max(),
            'best_f1':       df['pooled_macro_f1'].max(),
            'best_kappa':    df['pooled_kappa'].max(),
            'best_mcc':      df['pooled_mcc'].max(),
            'fastest_tool':           df.loc[df['runtime_mean'].idxmin(), 'tool'],
            'most_memory_efficient':  df.loc[df['peak_system_memory_mean_mb'].idxmin(), 'tool']
                                      if not df['peak_system_memory_mean_mb'].isna().all() else None
        }
    }


def run_all_tools_cv(adata: ad.AnnData, tools_to_run: List[str] = None,
                     cv_config: Dict = None) -> Dict[str, Dict]:
    if tools_to_run is None: tools_to_run = list(TOOL_REGISTRY.keys())
    if cv_config is None:    cv_config = CV_CONFIG
    print('=== CELL TYPE ANNOTATION BENCHMARKING ===')
    print(f'Dataset: {len(adata.obs)} cells x {len(adata.var)} genes')
    if 'Ground_Truth_Celltype' in adata.obs.columns:
        print(f'Cell types: {list(adata.obs["Ground_Truth_Celltype"].unique())}')
    print(f'Tools: {tools_to_run}')
    folds = create_cv_folds(adata, k=cv_config['k'], group_by=cv_config['group_by'],
                            stratify_by=cv_config['stratify_by'], seed=cv_config['seed'])
    results = {}
    for name in tools_to_run:
        fn = TOOL_REGISTRY.get(name)
        if fn is None:
            warnings.warn(f'Tool {name} not in registry'); continue
        results[name] = run_single_tool_cv(adata, fn, folds, name)
    return results


def run_all_tools_single_split(adata: ad.AnnData, tools_to_run: List[str] = None,
                               label_col: str = 'Ground_Truth_Celltype',
                               seed: int = 123) -> Dict[str, Dict]:
    if tools_to_run is None: tools_to_run = list(TOOL_REGISTRY.keys())
    print('=== BENCHMARKING (Single 80/20 Split) ===')
    print(f'Dataset: {len(adata.obs)} cells x {len(adata.var)} genes')
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    valid_mask = adata.obs[label_col].notna()
    adata = adata[valid_mask].copy()
    train_idx, test_idx = next(skf.split(np.arange(len(adata.obs)), adata.obs[label_col]))
    adata_train = adata[train_idx].copy()
    adata_test  = adata[test_idx].copy()
    print(f'Train: {adata_train.n_obs} cells, Test: {adata_test.n_obs} cells')
    try:
        sc.tl.rank_genes_groups(adata_train, groupby=label_col, method='wilcoxon', use_raw=False)
        markers = sc.get.rank_genes_groups_df(adata_train, group=None)
    except Exception as e:
        warnings.warn(f'Marker detection failed: {e}'); markers = pd.DataFrame()
    results = {}
    for name in tools_to_run:
        fn = TOOL_REGISTRY.get(name)
        if fn is None:
            warnings.warn(f'Tool {name} not in registry'); continue
        print(f'\n=== Running {name} (single split) ===')
        try:
            t0 = time.perf_counter()
            res = fn(adata_train, adata_test, markers)
            wall = time.perf_counter() - t0
            runtime    = res.get('method_walltime', wall)
            peak_mem   = res.get('peak_memory_mb', None)
            peak_vram  = res.get('peak_vram_mb', None)
            preds      = res['predictions']
            trues      = res['true_labels']
            confs      = res.get('confidence_scores', [0.0]*len(preds))
            cids       = res.get('cell_ids', list(adata_test.obs.index))
            metrics    = calculate_metrics(preds, trues)
            pm         = dict(metrics); pm['confusion_matrix'] = None
            def _nan_ci(): return [np.nan, np.nan]
            fv = {
                'accuracy_mean': metrics['overall_accuracy'], 'accuracy_std': np.nan,
                'accuracy_folds': [metrics['overall_accuracy']], 'accuracy_ci': _nan_ci(),
                'f1_mean': metrics['macro_f1'], 'f1_std': np.nan,
                'f1_folds': [metrics['macro_f1']], 'f1_ci': _nan_ci(),
                'kappa_mean': metrics['cohens_kappa'], 'kappa_std': np.nan,
                'kappa_folds': [metrics['cohens_kappa']], 'kappa_ci': _nan_ci(),
                'mcc_mean': metrics['mcc'], 'mcc_std': np.nan, 'mcc_folds': [metrics['mcc']],
                'rare_f1_mean': metrics['rare_type_f1'], 'rare_f1_std': np.nan,
                'rare_f1_folds': [metrics['rare_type_f1']],
                'unassigned_mean': metrics['unassigned_rate'], 'unassigned_std': np.nan,
                'unassigned_folds': [metrics['unassigned_rate']]
            }
            vm = [peak_mem]  if peak_mem  is not None else []
            vv = [peak_vram] if peak_vram is not None and peak_vram > 0 else []
            results[name] = {
                'tool_name': name, 'pooled_metrics': pm, 'fold_variation': fv,
                'runtime_stats': {'mean_seconds': runtime, 'std_seconds': np.nan,
                                  'total_seconds': runtime, 'fold_runtimes': [runtime]},
                'memory_stats': {'mean_mb': peak_mem if peak_mem is not None else np.nan,
                                 'std_mb': np.nan, 'max_mb': peak_mem if peak_mem is not None else np.nan,
                                 'min_mb': peak_mem if peak_mem is not None else np.nan, 'fold_memories': vm},
                'vram_stats': {'mean_mb': peak_vram if peak_vram is not None and peak_vram > 0 else np.nan,
                               'std_mb': np.nan, 'max_mb': peak_vram if peak_vram is not None and peak_vram > 0 else np.nan,
                               'fold_vrams': vv},
                'detailed_results': {'predictions': preds, 'true_labels': trues,
                                     'confidence_scores': confs, 'cell_ids': cids},
                'num_successful_folds': 1, 'num_total_folds': 1
            }
            mem_s  = f', RAM: {peak_mem:.0f} MB'  if peak_mem  is not None else ''
            vram_s = f', VRAM: {peak_vram:.0f} MB' if peak_vram else ''
            print(f'    Done in {runtime:.2f}s, accuracy: {metrics["overall_accuracy"]*100:.2f}%{mem_s}{vram_s}')
        except Exception as e:
            warnings.warn(f'{name} failed: {e}'); print(f'    FAILED: {e}')
    return results


def run_separate_datasets(adata_train: ad.AnnData, adata_test: ad.AnnData,
                          tools_to_run: List[str] = None,
                          label_col: str = 'Ground_Truth_Celltype') -> Dict[str, Dict]:
    if tools_to_run is None: tools_to_run = list(TOOL_REGISTRY.keys())
    print('=== BENCHMARKING (Separate Datasets) ===')
    print(f'Train: {adata_train.n_obs} x {adata_train.n_vars} | Test: {adata_test.n_obs} x {adata_test.n_vars}')
    try:
        sc.tl.rank_genes_groups(adata_train, groupby=label_col, method='wilcoxon', use_raw=False)
        markers = sc.get.rank_genes_groups_df(adata_train, group=None)
    except Exception as e:
        warnings.warn(f'Marker detection failed: {e}'); markers = pd.DataFrame()
    results = {}
    for name in tools_to_run:
        fn = TOOL_REGISTRY.get(name)
        if fn is None:
            warnings.warn(f'Tool {name} not in registry'); continue
        print(f'\n=== Running {name} (separate datasets) ===')
        try:
            t0 = time.perf_counter()
            res = fn(adata_train, adata_test, markers)
            wall = time.perf_counter() - t0
            runtime   = res.get('method_walltime', wall)
            peak_mem  = res.get('peak_memory_mb', None)
            peak_vram = res.get('peak_vram_mb', None)
            preds     = res['predictions']
            trues     = res['true_labels']
            confs     = res.get('confidence_scores', [0.0]*len(preds))
            cids      = res.get('cell_ids', list(adata_test.obs.index))
            metrics   = calculate_metrics(preds, trues)
            pm        = dict(metrics); pm['confusion_matrix'] = None
            def _nan_ci(): return [np.nan, np.nan]
            fv = {
                'accuracy_mean': metrics['overall_accuracy'], 'accuracy_std': np.nan,
                'accuracy_folds': [metrics['overall_accuracy']], 'accuracy_ci': _nan_ci(),
                'f1_mean': metrics['macro_f1'], 'f1_std': np.nan,
                'f1_folds': [metrics['macro_f1']], 'f1_ci': _nan_ci(),
                'kappa_mean': metrics['cohens_kappa'], 'kappa_std': np.nan,
                'kappa_folds': [metrics['cohens_kappa']], 'kappa_ci': _nan_ci(),
                'mcc_mean': metrics['mcc'], 'mcc_std': np.nan, 'mcc_folds': [metrics['mcc']],
                'rare_f1_mean': metrics['rare_type_f1'], 'rare_f1_std': np.nan,
                'rare_f1_folds': [metrics['rare_type_f1']],
                'unassigned_mean': metrics['unassigned_rate'], 'unassigned_std': np.nan,
                'unassigned_folds': [metrics['unassigned_rate']]
            }
            vm = [peak_mem]  if peak_mem  is not None else []
            vv = [peak_vram] if peak_vram is not None and peak_vram > 0 else []
            results[name] = {
                'tool_name': name, 'pooled_metrics': pm, 'fold_variation': fv,
                'runtime_stats': {'mean_seconds': runtime, 'std_seconds': np.nan,
                                  'total_seconds': runtime, 'fold_runtimes': [runtime]},
                'memory_stats': {'mean_mb': peak_mem if peak_mem is not None else np.nan,
                                 'std_mb': np.nan, 'max_mb': peak_mem if peak_mem is not None else np.nan,
                                 'min_mb': peak_mem if peak_mem is not None else np.nan, 'fold_memories': vm},
                'vram_stats': {'mean_mb': peak_vram if peak_vram is not None and peak_vram > 0 else np.nan,
                               'std_mb': np.nan, 'max_mb': peak_vram if peak_vram is not None and peak_vram > 0 else np.nan,
                               'fold_vrams': vv},
                'detailed_results': {'predictions': preds, 'true_labels': trues,
                                     'confidence_scores': confs, 'cell_ids': cids},
                'num_successful_folds': 1, 'num_total_folds': 1
            }
            mem_s  = f', RAM: {peak_mem:.0f} MB'  if peak_mem  is not None else ''
            vram_s = f', VRAM: {peak_vram:.0f} MB' if peak_vram else ''
            print(f'    Done in {runtime:.2f}s, accuracy: {metrics["overall_accuracy"]*100:.2f}%{mem_s}{vram_s}')
        except Exception as e:
            warnings.warn(f'{name} failed: {e}'); print(f'    FAILED: {e}')
    return results


print("Helper functions loaded.")


In [ ]:
import cell2sentence as cs
from cell2sentence.tasks import predict_cell_types_of_data
from transformers import TrainingArguments


def run_C2S_function(adata_train: ad.AnnData, adata_test: ad.AnnData,
                     markers: pd.DataFrame) -> Dict[str, Any]:
    """
    Cell2Sentence (C2S) cell-type annotation.
    Runs directly in-process (no subprocess) — suitable for Google Colab.
    Uses original batch sizes: per_device_train/eval_batch_size=8,
    gradient_accumulation_steps=4, no gradient_checkpointing.
    """

    def default_return():
        n = len(adata_test.obs)
        return {
            'predictions':       ['Unknown'] * n,
            'true_labels':       list(adata_test.obs['Ground_Truth_Celltype']),
            'confidence_scores': [0.0] * n,
            'cell_ids':          list(adata_test.obs.index),
        }

    if 'Ground_Truth_Celltype' not in adata_train.obs.columns:
        warnings.warn('Ground_Truth_Celltype not found in training data')
        return default_return()
    if 'Ground_Truth_Celltype' not in adata_test.obs.columns:
        warnings.warn('Ground_Truth_Celltype not found in test data')
        return default_return()

    try:
        # Work on copies so we don't mutate the cached fold data
        adata_train = adata_train.copy()
        adata_test  = adata_test.copy()

        # C2S expects a 'cell_type' column
        adata_train.obs['cell_type'] = adata_train.obs['Ground_Truth_Celltype'].astype(str)
        adata_test.obs['cell_type']  = adata_test.obs['Ground_Truth_Celltype'].astype(str)

        # Add required metadata columns with placeholder values if absent
        for col, val in {'organism': 'Homo sapiens', 'tissue': 'unknown',
                         'sex': 'unknown', 'batch_condition': 'unknown'}.items():
            if col not in adata_train.obs.columns:
                adata_train.obs[col] = val
            if col not in adata_test.obs.columns:
                adata_test.obs[col] = val

        if torch.cuda.is_available():
            gpu_name   = torch.cuda.get_device_name(0)
            gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
            print(f'[C2S] Device: GPU ({gpu_name}, {gpu_mem_gb:.1f} GB VRAM)')
        else:
            print('[C2S] Device: CPU (no GPU detected)')

        _timing    = [0.0]
        _label_cols = ['cell_type', 'organism', 'tissue', 'sex', 'batch_condition']

        with tempfile.TemporaryDirectory() as temp_dir:

            def _run_method():
                if torch.cuda.is_available():
                    print('[C2S] Running with GPU acceleration.', flush=True)
                    torch.cuda.reset_peak_memory_stats()
                else:
                    print('[C2S] Running on CPU.', flush=True)

                _t0 = time.perf_counter()

                # Convert training data to cell sentences
                print('[C2S] Converting training data to cell sentences...', flush=True)
                train_arrow_ds, train_vocab = cs.CSData.adata_to_arrow(
                    adata=adata_train, random_state=1234,
                    sentence_delimiter=' ', label_col_names=_label_cols)
                train_csdata = cs.CSData.csdata_from_arrow(
                    arrow_dataset=train_arrow_ds, vocabulary=train_vocab,
                    save_dir=temp_dir, save_name='c2s_train', dataset_backend='arrow')

                # Convert test data to cell sentences
                print('[C2S] Converting test data to cell sentences...', flush=True)
                test_arrow_ds, test_vocab = cs.CSData.adata_to_arrow(
                    adata=adata_test, random_state=1234,
                    sentence_delimiter=' ', label_col_names=_label_cols)
                test_csdata = cs.CSData.csdata_from_arrow(
                    arrow_dataset=test_arrow_ds, vocabulary=test_vocab,
                    save_dir=temp_dir, save_name='c2s_test', dataset_backend='arrow')

                # Load pretrained model from HuggingFace Hub
                print('[C2S] Loading pretrained model...', flush=True)
                csmodel = cs.CSModel(
                    model_name_or_path='vandijklab/C2S-Pythia-410m-cell-type-prediction',
                    save_dir=temp_dir, save_name='c2s_model')

                # Fine-tune on training data (original batch sizes)
                print('[C2S] Fine-tuning on training data...', flush=True)
                output_dir = os.path.join(temp_dir, 'c2s_output')
                os.makedirs(output_dir, exist_ok=True)
                train_args = TrainingArguments(
                    output_dir=output_dir,
                    num_train_epochs=5,
                    learning_rate=1e-5,
                    per_device_train_batch_size=8,
                    per_device_eval_batch_size=8,
                    gradient_accumulation_steps=4,
                    bf16=True,
                    fp16=False,
                    lr_scheduler_type='cosine',
                    warmup_ratio=0.05,
                    logging_steps=50,
                    logging_strategy='steps',
                    eval_steps=50,
                    eval_strategy='steps',
                    save_strategy='no',
                    report_to='none',
                    load_best_model_at_end=False,
                )
                csmodel.fine_tune(
                    csdata=train_csdata,
                    task='cell_type_prediction',
                    train_args=train_args,
                    loss_on_response_only=True,
                    top_k_genes=200,
                    max_eval_samples=500,
                )

                # Predict cell types on test data
                print('[C2S] Predicting cell types...', flush=True)
                raw_predictions = predict_cell_types_of_data(
                    csdata=test_csdata, csmodel=csmodel, n_genes=200)

                _timing[0] = time.perf_counter() - _t0
                peak_vram_bytes = torch.cuda.max_memory_allocated() if torch.cuda.is_available() else 0
                return raw_predictions, peak_vram_bytes / (1024 * 1024)

            mem_list, (raw_predictions, peak_vram_mb) = memory_usage(
                (_run_method, [], {}), interval=0.1, include_children=True, retval=True)

        peak_system_memory_mb = max(mem_list)
        method_walltime       = _timing[0]

        predictions   = [str(p).rstrip('.').strip() if p else 'Unknown' for p in raw_predictions]
        true_labels   = list(adata_test.obs['Ground_Truth_Celltype'])
        cell_ids      = list(adata_test.obs.index)
        conf_scores   = [0.0 if p == 'Unknown' else 0.8 for p in predictions]

        print(f'[C2S] Done: {len(predictions)} predictions, '
              f'{len(set(predictions))} unique types, '
              f'{sum(1 for p in predictions if p == "Unknown")} unknown')

        return {
            'predictions':       predictions,
            'true_labels':       true_labels,
            'confidence_scores': conf_scores,
            'cell_ids':          cell_ids,
            'method_walltime':   method_walltime,
            'peak_memory_mb':    peak_system_memory_mb,
            'peak_vram_mb':      peak_vram_mb,
        }

    except Exception as e:
        warnings.warn(f'C2S error: {e}')
        import traceback; traceback.print_exc()
        return default_return()


print("C2S tool function defined.")


In [ ]:
adata = sc.read_h5ad(DATA_PATH)
print(adata)
print(f"Cell types: {list(adata.obs['Ground_Truth_Celltype'].value_counts().index)}")


In [ ]:
# Tool registry — add more tools here if needed
TOOL_REGISTRY = {'C2S': run_C2S_function}

# ── Choose execution mode ───────────────────────────────────────────────────

# MODE 1: K-fold cross-validation (default, recommended)
#results = run_all_tools_cv(adata, tools_to_run=['C2S'], cv_config=CV_CONFIG)

# MODE 2: Single 80/20 stratified split (uncomment to use instead)
results = run_all_tools_single_split(adata, tools_to_run=['C2S'])

# MODE 3: Separate train/test datasets (uncomment to use instead)
# adata_train = sc.read_h5ad('/content/drive/MyDrive/path/to/train.h5ad')
# adata_test  = sc.read_h5ad('/content/drive/MyDrive/path/to/test.h5ad')
# results = run_separate_datasets(adata_train, adata_test, tools_to_run=['C2S'])

# ── Aggregate and display ───────────────────────────────────────────────────
final = aggregate_all_tools(results)
print("\n=== Results ===")
print(final['comparison_table'].to_string(index=False))


In [ ]:
# Save results to Google Drive
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

pkl_path = os.path.join(RESULTS_DIR, f'c2s_results_{timestamp}.pkl')
tsv_path = os.path.join(RESULTS_DIR, f'Marques-Brain-2016_for_use_subset_c2s_results_{timestamp}.tsv')

with open(pkl_path, 'wb') as f:
    pickle.dump(results, f)

final['comparison_table'].to_csv(tsv_path, sep='\t', index=False)

print(f"Saved pickle: {pkl_path}")
print(f"Saved TSV:    {tsv_path}")

# Quick summary
print("\n=== Summary ===")
t = final['comparison_table']
for _, row in t.iterrows():
    print(f"  {row['tool']}: accuracy={row['pooled_accuracy']:.3f}, "
          f"macro_F1={row['pooled_macro_f1']:.3f}, kappa={row['pooled_kappa']:.3f}, "
          f"runtime={row['runtime_mean']:.1f}s")

In [ ]:
print(results)

In [ ]:
from google.colab import runtime
runtime.unassign()